# XGBoost spatial-support benchmark: 100 m, 150 m and 200 m

This verification notebook launches or reuses the isolated benchmark for the clear 22 June 2026 Landsat observation. All three models use one common cohort complete throughout 200 m, identical sector-grouped folds, a 400 m embargo, nested spatial tuning and leakage-safe backward feature elimination. The production scenario booster is not modified.

In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from greenwave_local_layers.image_regression_radius_benchmark import (
    PREDICTIONS_PATH, REPORT_PATH, run_radius_benchmark,
)

## Run or reuse the benchmark

In [2]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    report = run_radius_benchmark(device='cuda')
else:
    report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
pd.Series({
    'common observations': report['commonCohort']['sampleCount'],
    'Statbel sectors': report['commonCohort']['sectorCount'],
    'outer folds': report['folds']['outerFoldCount'],
    'inner folds': report['folds']['innerFoldCount'],
    'embargo (m)': report['folds']['embargoMeters'],
    'training device': report['training']['device'],
}).to_frame('value')

,value
common observations,186449
Statbel sectors,154
outer folds,5
inner folds,4
embargo (m),400
training device,cuda


## Common cohort and exact feature prefixes

In [3]:
pd.DataFrame([{
    'radius_m': int(radius),
    'predictors': item['featureCount'],
    'pooled_rmse_c': item['pooledOuterMetrics']['rmse_c'],
    'pooled_mae_c': item['pooledOuterMetrics']['mae_c'],
    'pooled_r2': item['pooledOuterMetrics']['r2'],
    'mean_error_c': item['pooledOuterMetrics']['mean_error_c'],
} for radius, item in report['radii'].items()]).sort_values('radius_m')

,radius_m,predictors,pooled_rmse_c,pooled_mae_c,pooled_r2,mean_error_c
0,100,20,2.513748,1.904663,0.596722,-0.100691
1,150,30,2.502994,1.883890,0.600165,-0.074759
2,200,40,2.463719,1.860538,0.612614,-0.074434


## Per-fold held-out performance

In [4]:
fold_rows = []
for radius, item in report['radii'].items():
    for fold in item['outerFolds']:
        fold_rows.append({'radius_m': int(radius), 'fold': fold['fold'], **fold['metrics']})
fold_metrics = pd.DataFrame(fold_rows)
fold_metrics.sort_values(['radius_m', 'fold'])

,radius_m,fold,count,mae_c,rmse_c,r2,mean_error_c
0,100,0,37284,1.612082,2.094926,0.609466,-0.244622
1,100,1,37292,1.827847,2.298831,0.559090,0.441330
2,100,2,37298,1.750921,2.186573,0.579962,0.132327
3,100,3,37292,2.333545,3.317032,0.601770,0.448495
4,100,4,37283,1.998905,2.477354,0.593670,-1.281338
5,150,0,37284,1.587401,2.047957,0.626781,-0.247931
6,150,1,37292,1.812616,2.287918,0.563266,0.509671
7,150,2,37298,1.712166,2.140136,0.597613,0.140605
8,150,3,37292,2.308325,3.320251,0.600997,0.493883
9,150,4,37283,1.998936,2.508342,0.583441,-1.270385


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for radius, group in fold_metrics.groupby('radius_m'):
    axes[0].plot(group.fold, group.rmse_c, marker='o', label=f'{radius} m')
    axes[1].plot(group.fold, group.mae_c, marker='o', label=f'{radius} m')
    axes[2].plot(group.fold, group.r2, marker='o', label=f'{radius} m')
axes[0].set(xlabel='Outer fold', ylabel='RMSE (Ã‚Â°C)', title='Held-out RMSE')
axes[1].set(xlabel='Outer fold', ylabel='MAE (Ã‚Â°C)', title='Held-out MAE')
axes[2].set(xlabel='Outer fold', ylabel='RÃ‚Â²', title='Held-out explained variation')
for axis in axes: axis.legend()
fig.tight_layout()

## Shared fold composition and every tested recipe

In [6]:
pd.DataFrame([{
    'outer_fold': fold['fold'],
    'training observations': fold['diagnostics']['trainSampleCount'],
    'held-out observations': fold['diagnostics']['testSampleCount'],
    'embargoed observations': fold['diagnostics']['excludedBufferSampleCount'],
    'training sectors': fold['diagnostics']['trainSectorCount'],
    'held-out sectors': fold['diagnostics']['testSectorCount'],
} for fold in report['folds']['outer']])

,outer_fold,training observations,held-out observations,embargoed observations,training sectors,held-out sectors
0,0,102098,37284,47067,111,29
1,1,106639,37292,42518,105,31
2,2,100035,37298,49116,104,32
3,3,104492,37292,44665,108,31
4,4,106338,37283,42828,109,31


In [7]:
configuration_rows = []
for radius, radius_report in report['radii'].items():
    for outer in radius_report['outerFolds']:
        for tested in outer['testedConfigurations']:
            configuration_rows.append({
                'radius_m': int(radius), 'outer_fold': outer['fold'],
                'spatial_rmse_c': tested['meanSpatialRmseC'],
                'best_rounds': tested['bestRounds'], **tested['parameters'],
            })
pd.DataFrame(configuration_rows).sort_values(['radius_m', 'outer_fold', 'spatial_rmse_c'])

,radius_m,outer_fold,spatial_rmse_c,best_rounds,num_boost_round,early_stopping_rounds,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,seed
0,100,0,2.684851,"[234, 92, 31, 44]",2000,60,0.05,6,5,0.80,0.70,0.10,4,42
1,100,0,2.685378,"[494, 207, 58, 77]",2000,60,0.03,5,3,0.85,0.70,0.05,2,42
2,100,0,2.694266,"[264, 106, 21, 30]",2000,60,0.08,4,5,0.75,0.85,0.10,2,42
3,100,0,2.696565,"[314, 139, 36, 51]",2000,60,0.05,4,1,0.80,1.00,0.05,1,42
4,100,0,2.706720,"[803, 186, 40, 53]",2000,60,0.05,3,3,1.00,0.85,0.00,2,42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,200,4,2.406473,"[47, 29, 69, 26]",2000,60,0.10,3,1,0.70,0.70,0.20,4,42
116,200,4,2.414356,"[40, 34, 55, 37]",2000,60,0.08,4,5,0.75,0.85,0.10,2,42
117,200,4,2.422928,"[48, 54, 123, 60]",2000,60,0.05,4,1,0.80,1.00,0.05,1,42
118,200,4,2.449167,"[88, 79, 169, 96]",2000,60,0.03,5,3,0.85,0.70,0.05,2,42


In [8]:
selection_rows = []
for radius, radius_report in report['radii'].items():
    for decision in radius_report['final']['featureSelection']:
        selection_rows.append({'radius_m': int(radius), **decision})
pd.DataFrame(selection_rows).sort_values(['radius_m']).reset_index(drop=True)

,radius_m,feature,permutationImportanceC,beforeRmseC,afterRmseC,removed
0,100,low_green_75_100m,-0.010812,2.506176,2.504564,True
1,100,agriculture_50_75m,-0.007502,2.504564,2.500993,True
2,100,low_green_0_25m,-0.001436,2.500993,2.504852,True
3,100,low_green_50_75m,-0.003539,2.504852,2.504145,True
4,100,water_75_100m,-0.000807,2.504145,2.502833,True
5,100,water_50_75m,-0.000819,2.502833,2.500235,True
6,100,water_25_50m,-0.000881,2.500235,2.499266,True
7,100,water_0_25m,0.000002,2.499266,2.499574,True
8,100,low_green_25_50m,0.003196,2.499574,2.496350,True
9,100,high_green_0_25m,-0.002611,2.496350,2.499191,True


## Selected parameters and retained predictors

In [9]:
pd.DataFrame([{
    'radius_m': int(radius),
    'retained_predictors': len(item['final']['retainedFeatures']),
    'features': ', '.join(item['final']['retainedFeatures']),
    'rounds': item['final']['boostRounds'],
    'spatial_cv_rmse_c': item['final']['spatialCvRmseC'],
    **item['final']['parameters'],
} for radius, item in report['radii'].items()]).sort_values('radius_m')

,radius_m,retained_predictors,features,rounds,spatial_cv_rmse_c,num_boost_round,early_stopping_rounds,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,seed
0,100,4,"soil_sealing_25_50m, soil_sealing_50_75m, soil...",109,2.474992,2000,60,0.03,5,3,0.85,0.70,0.05,2,42
1,150,24,"soil_sealing_0_25m, high_green_0_25m, low_gree...",48,2.456309,2000,60,0.08,4,5,0.75,0.85,0.10,2,42
2,200,29,"soil_sealing_0_25m, high_green_0_25m, low_gree...",136,2.423183,2000,60,0.03,5,3,0.85,0.70,0.05,2,42


## Paired sector-bootstrap comparisons

In [10]:
comparison_rows = []
for label, item in report['pairedBootstrap']['comparisons'].items():
    comparison_rows.append({
        'comparison': label,
        'RMSE improvement (Ã‚Â°C)': item['rmseImprovementC'],
        'RMSE improvement (%)': item['rmseImprovementPercent'],
        'RMSE delta 95% interval': item['metrics']['rmse_c']['ci95'],
        'MAE delta 95% interval': item['metrics']['mae_c']['ci95'],
        'RÃ‚Â² delta 95% interval': item['metrics']['r2']['ci95'],
        'supported improvement': item['statisticallySupportedRmseImprovement'],
    })
pd.DataFrame(comparison_rows)

,comparison,RMSE improvement (Ã‚Â°C),RMSE improvement (%),RMSE delta 95% interval,MAE delta 95% interval,RÃ‚Â² delta 95% interval,supported improvement
0,150m-vs-100m,0.010754,0.427794,"[-0.054481254139398264, 0.03713718285105653]","[-0.04762677891807753, 0.009850664279205576]","[-0.01192156577541564, 0.018499594578269246]",False
1,200m-vs-100m,0.050029,1.990201,"[-0.10205941160339176, 0.004706040778152807]","[-0.08519284128024991, -0.006826547657623669]","[-0.0016194308497965906, 0.030514377727166014]",False
2,200m-vs-150m,0.039275,1.569119,"[-0.08580122947970514, 0.003810192732965759]","[-0.05490177593816342, 0.003900769102494205]","[-0.0013077952465949448, 0.02465423522750186]",False


## Observed-versus-predicted and residual diagnostics

In [11]:
held_out = np.load(PREDICTIONS_PATH)
observed = held_out['observed_c']
fig, axes = plt.subplots(3, 2, figsize=(11, 13))
for row, radius in enumerate((100, 150, 200)):
    predicted = held_out[f'predicted_{radius}m_c']
    residual = predicted - observed
    axes[row, 0].hexbin(observed, predicted, gridsize=45, mincnt=1, cmap='viridis')
    limits = [min(observed.min(), predicted.min()), max(observed.max(), predicted.max())]
    axes[row, 0].plot(limits, limits, '--', color='black', linewidth=1)
    axes[row, 0].set(xlabel='Observed LST (Ã‚Â°C)', ylabel='Held-out predicted LST (Ã‚Â°C)', title=f'{radius} m observed vs predicted')
    axes[row, 1].hexbin(predicted, residual, gridsize=45, mincnt=1, cmap='magma')
    axes[row, 1].axhline(0, color='black', linestyle='--', linewidth=1)
    axes[row, 1].set(xlabel='Held-out predicted LST (Ã‚Â°C)', ylabel='Predicted Ã¢Ë†â€™ observed (Ã‚Â°C)', title=f'{radius} m residuals')
fig.tight_layout()

The 20,000 paired resamples treat each statistical sector as the resampling unit, preserving within-sector dependence and comparing radii on identical held-out observations. A candidate is labelled better only when its candidate-minus-baseline RMSE interval lies entirely below zero. These are observational prediction results, not counterfactual uncertainty or causal evidence.